# Experiment 16 — Noise-floor estimation

## Setup

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import negolin_utils as nu
nu.setup_matplotlib()

# Override paths if your data lives elsewhere:
# nu.RAND_BUYER_DIR = Path('/abs/path/to/0421_redo_previous_exp/results')

# Load data and apply v7 preprocessing
raw = nu.load_all_fixed_policies(nu.RAND_BUYER_DIR)
_, turns = nu.preprocess_turns(raw)
POLICIES = list(nu.POLICIES)

# Load Stage-2 linear model selection from Exp02 (if present)
try:
    selected = pd.read_csv('selected_linear_models.csv')
    SELECTED = {row['policy']: (row['feature'], row['model'], row.get('alpha', np.nan))
                for _, row in selected.iterrows()}
    print('Loaded per-policy (feature, model, alpha) from Exp02.')
except FileNotFoundError:
    # Fall back to a reasonable default (Feature II, Ridge) if Exp02 hasn't run
    SELECTED = {p: ('II', 'Ridge', 1.0) for p in POLICIES}
    print('selected_linear_models.csv not found; using default (Feature II, Ridge).')

**Goal.** Estimate the irreducible noise in each LLM seller policy by
replaying a held-out state many times and measuring the within-state
standard deviation of the seller's ask. The model error can never go below
this noise floor.

**Data requirement.** For ~100 held-out states per policy, 10 replays each,
saved as `../data/noise_floor/noise_floor_results.json`.

In [ ]:
from pathlib import Path
import json
NF_PATH = Path('../data/noise_floor/noise_floor_results.json')

if not NF_PATH.exists():
    # Emit sampling spec and stop
    sampled = []
    rng = np.random.default_rng(42)
    clean = turns[~turns['parse_fail']]
    for p in POLICIES:
        pol = clean[clean['policy'] == p]
        if pol.empty: continue
        keys = pol[['run_id','round','car_name']].drop_duplicates()
        k = min(20, len(keys))
        sampled.append(keys.sample(n=k, random_state=0).assign(policy=p))
    spec = pd.concat(sampled, ignore_index=True)
    spec.to_csv('exp16_noise_spec.csv', index=False)
    print(f'Wrote {len(spec)} sampling states to exp16_noise_spec.csv.')
    print('Run the replay worker to produce noise_floor_results.json, then re-run.')
else:
    data = json.loads(NF_PATH.read_text())
    sd_records = []
    for state_id, replays in data.items():
        sd_records.append({'state_id': state_id,
                            'policy': replays[0]['policy'],
                            'sd': float(np.std([r['S_norm'] for r in replays]))})
    sd_df = pd.DataFrame(sd_records)
    print(sd_df.groupby('policy')['sd'].describe().round(4))